# nb02 - Tableau to SQL to pandas parity checks

**Purpose:** every number in the blog post *Comparing Medi-Cal Health Plan Quality in Tableau* is reproduced here in **DuckDB SQL** and **pandas** and asserted to match the value shown in Tableau. If this notebook runs top to bottom with no `AssertionError`, all parity claims in the post hold.

Tableau values are recorded as constants (`TABLEAU`) from the built workbook; SQL and pandas recompute them from the same cleaned CSVs produced by `nb01`.

Sections: 1) statewide average benchmark (FIXED LOD), 2) FIXED reference line, 3) change since 2016 (Difference From First), 4) year parameter ranking, 5) dashboard companions.

In [1]:
import pandas as pd
import duckdb

DATA = '../data'

# aqfs_clean holds one score per reporting unit per year, PLUS a derived
# 'Statewide Average' row per year (built in nb01). Rename AQFS -> Aqfs to match
# the Tableau field name used in the blog snippets.
df = pd.read_csv(f'{DATA}/aqfs_clean.csv').rename(columns={'AQFS': 'Aqfs'})

plans  = df[df.Plan != 'Statewide Average'].copy()   # real reporting units only
sw_row = df[df.Plan == 'Statewide Average'].copy()   # the derived benchmark row

con = duckdb.connect()
con.register('plans', plans)
con.register('df', df)

print('rows:', len(df), '| real units:', len(plans), '| derived sw rows:', len(sw_row))
print('years:', sorted(plans.Year.unique()))

CHECKS = []
def record(name, tableau, sql, pdv):
    ok = (round(float(sql), 2) == round(float(tableau), 2) == round(float(pdv), 2))
    CHECKS.append((name, tableau, round(float(sql), 2), round(float(pdv), 2), ok))
    assert ok, f'PARITY FAIL: {name}: tableau={tableau} sql={sql} pandas={pdv}'
    print(f'OK  {name:40s} tableau={tableau}  sql={round(float(sql),2)}  pandas={round(float(pdv),2)}')

rows: 444 | real units: 436 | derived sw rows: 8
years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


## Section 1 - Benchmark: the statewide average per year

`{ FIXED [Year] : AVG([Aqfs]) }` in Tableau, `AVG(Aqfs) OVER (PARTITION BY Year)` in SQL, `groupby('Year').transform('mean')` in pandas. Blog values: 2016 = 55.45, 2020 = 62.67, 2023 = 57.41.

In [2]:
TABLEAU_SW = {2016: 55.45, 2020: 62.67, 2023: 57.41}

sql_sw = con.execute('''
    SELECT Year, ROUND(AVG(Aqfs), 2) AS sw_avg
    FROM plans GROUP BY Year ORDER BY Year
''').df().set_index('Year')['sw_avg']

pd_sw = plans.groupby('Year')['Aqfs'].mean().round(2)

for y, tv in TABLEAU_SW.items():
    record(f'statewide average {y}', tv, sql_sw[y], pd_sw[y])

OK  statewide average 2016                   tableau=55.45  sql=55.45  pandas=55.45
OK  statewide average 2020                   tableau=62.67  sql=62.67  pandas=62.67
OK  statewide average 2023                   tableau=57.41  sql=57.41  pandas=57.41


## Section 2 - FIXED reference line value (2023)

Filtered to 2023, the reference line sits at the statewide average, 57.41.

In [3]:
TABLEAU_REF_2023 = 57.41
sql_ref = con.execute("SELECT ROUND(AVG(Aqfs),2) FROM plans WHERE Year = 2023").fetchone()[0]
pd_ref  = plans.loc[plans.Year == 2023, 'Aqfs'].mean()
record('FIXED reference line 2023', TABLEAU_REF_2023, sql_ref, pd_ref)

OK  FIXED reference line 2023                tableau=57.41  sql=57.41  pandas=57.41


## Section 3 - Change since 2016 (Difference From First)

`Aqfs - FIRST_VALUE(Aqfs) OVER (PARTITION BY unit ORDER BY Year)` in SQL, `x - x.iloc[0]` per group in pandas. Blog: L.A. Care 2020 peak +11.31, L.A. Care net -1.58, Health Net net -9.36, statewide net +1.96.

In [4]:
plans_sorted = plans.sort_values(['Reporting Unit', 'Year'])
plans_sorted['change_since_2016'] = (plans_sorted.groupby('Reporting Unit')['Aqfs']
                                     .transform(lambda s: s - s.iloc[0]))
def pd_change(unit, year):
    m = plans_sorted[(plans_sorted['Reporting Unit'] == unit) & (plans_sorted.Year == year)]
    return float(m['change_since_2016'].iloc[0])

sw = sw_row.sort_values('Year').set_index('Year')['Aqfs']
pd_sw_change_2023 = float(sw[2023] - sw[2016])

sql_change = con.execute('''
    WITH t AS (
        SELECT Year, "Reporting Unit" AS unit, Aqfs,
               Aqfs - FIRST_VALUE(Aqfs) OVER (PARTITION BY "Reporting Unit" ORDER BY Year) AS chg
        FROM plans)
    SELECT unit, Year, ROUND(chg,2) AS chg FROM t
    WHERE unit IN ('LA Care - Los Angeles','Health Net - Los Angeles')
''').df()
def sql_change_v(unit, year):
    return float(sql_change[(sql_change.unit==unit)&(sql_change.Year==year)].chg.iloc[0])

sql_sw_change_2023 = con.execute('''
    SELECT ROUND(MAX(CASE WHEN Year=2023 THEN a END) - MAX(CASE WHEN Year=2016 THEN a END),2)
    FROM (SELECT Year, AVG(Aqfs) a FROM df WHERE Plan='Statewide Average' GROUP BY Year)
''').fetchone()[0]

record('LA Care change 2020 (peak)', 11.31, sql_change_v('LA Care - Los Angeles',2020), pd_change('LA Care - Los Angeles',2020))
record('LA Care net 2016->2023',    -1.58, sql_change_v('LA Care - Los Angeles',2023), pd_change('LA Care - Los Angeles',2023))
record('Health Net net 2016->2023', -9.36, sql_change_v('Health Net - Los Angeles',2023), pd_change('Health Net - Los Angeles',2023))
record('Statewide net 2016->2023',   1.96, sql_sw_change_2023, pd_sw_change_2023)

OK  LA Care change 2020 (peak)               tableau=11.31  sql=11.31  pandas=11.31
OK  LA Care net 2016->2023                   tableau=-1.58  sql=-1.58  pandas=-1.58
OK  Health Net net 2016->2023                tableau=-9.36  sql=-9.36  pandas=-9.36
OK  Statewide net 2016->2023                 tableau=1.96  sql=1.96  pandas=1.96


## Section 4 - Year parameter: the 2023 ranking

`WHERE Year = :year` then `RANK() OVER (ORDER BY Aqfs DESC)`. Blog: 56 units, top CCAH - Monterey/Santa Cruz at 90.0, L.A. Care ranked 26.

In [5]:
sql_rank = con.execute('''
    SELECT "Reporting Unit" AS unit, Aqfs, RANK() OVER (ORDER BY Aqfs DESC) AS rnk
    FROM plans WHERE Year = 2023
''').df()
sql_n = len(sql_rank)
sql_top_val = float(sql_rank.sort_values('rnk').iloc[0].Aqfs)
sql_la_rank = int(sql_rank[sql_rank.unit=='LA Care - Los Angeles'].rnk.iloc[0])

r = plans[plans.Year==2023].copy()
r['rnk'] = r.Aqfs.rank(ascending=False, method='min').astype(int)
pd_n = len(r)
pd_top_val = float(r.sort_values('rnk').iloc[0].Aqfs)
pd_la_rank = int(r[r['Reporting Unit']=='LA Care - Los Angeles'].rnk.iloc[0])

record('2023 units ranked',     56,   sql_n,       pd_n)
record('2023 top score (CCAH)', 90.0, sql_top_val, pd_top_val)
record('2023 LA Care rank',     26,   sql_la_rank, pd_la_rank)

OK  2023 units ranked                        tableau=56  sql=56.0  pandas=56.0
OK  2023 top score (CCAH)                    tableau=90.0  sql=90.0  pandas=90.0
OK  2023 LA Care rank                        tableau=26  sql=26.0  pandas=26.0


## Section 5 - Dashboard companions

Primary care access (mean PCPs per 2,000 members, L.A. Care = 3.75) and grievances (Quality of Service = 216,256 of 595,072 total, 2023 statewide).

In [6]:
pr = pd.read_csv(f'{DATA}/provider_ratios_clean.csv')
gr = pd.read_csv(f'{DATA}/grievance_type_clean.csv')
con.register('pr', pr); con.register('gr', gr)

sql_pcp = con.execute('''SELECT ROUND(AVG("PCPs per 2,000 Members"),2) FROM pr WHERE Plan='LA Care' ''').fetchone()[0]
pd_pcp  = pr.loc[pr.Plan=='LA Care', 'PCPs per 2,000 Members'].mean()
record('LA Care PCPs per 2,000', 3.75, sql_pcp, pd_pcp)

sql_qos = con.execute('''SELECT SUM(Grievances) FROM gr WHERE "Grievance Type Roll-up"='Quality of Service' ''').fetchone()[0]
pd_qos  = gr.loc[gr['Grievance Type Roll-up']=='Quality of Service','Grievances'].sum()
record('Grievances: Quality of Service', 216256, sql_qos, pd_qos)

sql_tot = con.execute('SELECT SUM(Grievances) FROM gr').fetchone()[0]
pd_tot  = gr.Grievances.sum()
record('Grievances: total 2023', 595072, sql_tot, pd_tot)

OK  LA Care PCPs per 2,000                   tableau=3.75  sql=3.75  pandas=3.75
OK  Grievances: Quality of Service           tableau=216256  sql=216256.0  pandas=216256.0
OK  Grievances: total 2023                   tableau=595072  sql=595072.0  pandas=595072.0


## Summary

Each row compares the Tableau value with the SQL and pandas recomputation; `pass` is True only when all three agree to two decimals.

In [7]:
summary = pd.DataFrame(CHECKS, columns=['check','tableau','sql','pandas','pass'])
print(summary.to_string(index=False))
assert summary['pass'].all(), 'At least one parity check failed'
print(f"\nALL {len(summary)} PARITY CHECKS PASSED")

                         check   tableau       sql    pandas  pass
        statewide average 2016     55.45     55.45     55.45  True
        statewide average 2020     62.67     62.67     62.67  True
        statewide average 2023     57.41     57.41     57.41  True
     FIXED reference line 2023     57.41     57.41     57.41  True
    LA Care change 2020 (peak)     11.31     11.31     11.31  True
        LA Care net 2016->2023     -1.58     -1.58     -1.58  True
     Health Net net 2016->2023     -9.36     -9.36     -9.36  True
      Statewide net 2016->2023      1.96      1.96      1.96  True
             2023 units ranked     56.00     56.00     56.00  True
         2023 top score (CCAH)     90.00     90.00     90.00  True
             2023 LA Care rank     26.00     26.00     26.00  True
        LA Care PCPs per 2,000      3.75      3.75      3.75  True
Grievances: Quality of Service 216256.00 216256.00 216256.00  True
        Grievances: total 2023 595072.00 595072.00 595072.00  